# 07 — Web Application Security Testing

## What This Is
Web application security testing (DAST — Dynamic Application Security Testing) probes running applications for injection flaws, authentication weaknesses, and logic errors. OWASP Top 10 defines the most critical categories.

## Why It Matters
Web applications are the most exposed attack surface for most organisations. OWASP A01 (Broken Access Control), A02 (Cryptographic Failures), and A03 (Injection) have topped breach statistics for years.

## Where the Community Stands
Burp Suite dominates professional web testing. OWASP ZAP provides an open-source alternative. Modern appsec programmes combine automated DAST scanners with manual review (logic flaws resist automation).

## Authorized Testing Context
All vulnerability probing here is simulated against synthetic endpoints. Real application testing requires explicit written scope from the application owner. Bug bounty scopes define exactly which endpoints are in-scope.

In [ ]:
import re
import json
import hashlib
import urllib.parse
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

# OWASP Top 10 categories (2021)
OWASP_TOP10 = {
    'A01': 'Broken Access Control',
    'A02': 'Cryptographic Failures',
    'A03': 'Injection',
    'A04': 'Insecure Design',
    'A05': 'Security Misconfiguration',
    'A06': 'Vulnerable & Outdated Components',
    'A07': 'Identification & Authentication Failures',
    'A08': 'Software & Data Integrity Failures',
    'A09': 'Security Logging & Monitoring Failures',
    'A10': 'Server-Side Request Forgery',
}

for code_id, name in OWASP_TOP10.items():
    print(f'  {code_id}: {name}')

In [ ]:
@dataclass
class WebFinding:
    owasp_id: str
    name: str
    endpoint: str
    parameter: str
    payload: str
    evidence: str
    severity: str   # Critical/High/Medium/Low/Info
    cwe: str

class DASTScanner:
    """Simulated DAST scanner against synthetic web application responses."""

    # Simulated application responses for testing
    SIMULATED_RESPONSES = {
        '/login?user=admin\'--&pass=x': (
            200, 'Welcome, admin! Last login: 2024-01-01',
            "You have 1 new message"
        ),
        '/search?q=<script>alert(1)</script>': (
            200, 'Results for: <script>alert(1)</script>',  # reflected!
            ''
        ),
        '/api/user/100': (200, '{"id":100,"role":"admin","email":"admin@corp.com"}', ''),
        '/api/user/101': (200, '{"id":101,"role":"user","email":"bob@corp.com"}', ''),
        '/.env': (200, 'DB_PASSWORD=supersecret123\nSECRET_KEY=abc123', ''),
        '/admin/panel': (200, 'Admin Dashboard - Users: 4821', ''),
        '/api/ssrf?url=http://169.254.169.254/latest/meta-data': (200, 'ami-id: ami-12345', ''),
    }

    def __init__(self):
        self.findings: List[WebFinding] = []

    def _simulate_request(self, endpoint: str) -> Tuple[int, str]:
        return self.SIMULATED_RESPONSES.get(endpoint, (404, 'Not Found'))

    def check_sqli(self, base: str, param: str) -> None:
        payloads = ["' OR '1'='1", "'--", "' UNION SELECT null--"]
        for payload in payloads:
            endpoint = f'{base}?{param}={urllib.parse.quote(payload)}&pass=x'
            # Simulate: login with SQLi payload
            if 'login' in base:
                resp_endpoint = f'/login?user=admin\'--&pass=x'
                code_val, body = self._simulate_request(resp_endpoint)
                if code_val == 200 and 'Welcome' in body:
                    self.findings.append(WebFinding(
                        owasp_id='A03', name='SQL Injection',
                        endpoint=base, parameter=param, payload=payload,
                        evidence='Authentication bypassed — Welcome message returned',
                        severity='Critical', cwe='CWE-89'
                    ))
                    return

    def check_xss(self, base: str, param: str) -> None:
        payload = '<script>alert(1)</script>'
        endpoint = f'{base}?{param}={urllib.parse.quote(payload)}'
        resp_endpoint = f'/search?q=<script>alert(1)</script>'
        code_val, body = self._simulate_request(resp_endpoint)
        if payload in body and code_val == 200:
            self.findings.append(WebFinding(
                owasp_id='A03', name='Reflected XSS',
                endpoint=base, parameter=param, payload=payload,
                evidence='Payload reflected unencoded in response',
                severity='High', cwe='CWE-79'
            ))

    def check_idor(self, base: str, user_id: int) -> None:
        for test_id in [user_id - 1, user_id + 1, 1, 100]:
            endpoint = f'{base}/{test_id}'
            code_val, body = self._simulate_request(endpoint)
            if code_val == 200 and 'admin' in body.lower() and test_id != user_id:
                self.findings.append(WebFinding(
                    owasp_id='A01', name='IDOR (Broken Access Control)',
                    endpoint=base, parameter='id', payload=str(test_id),
                    evidence=f'Accessed admin record at id={test_id}: {body[:60]}',
                    severity='High', cwe='CWE-639'
                ))
                break

    def check_sensitive_files(self) -> None:
        targets = ['/.env', '/config.php', '/.git/config', '/backup.zip']
        for t in targets:
            code_val, body = self._simulate_request(t)
            if code_val == 200 and any(kw in body for kw in ['PASSWORD','SECRET','password','secret']):
                self.findings.append(WebFinding(
                    owasp_id='A05', name='Sensitive File Exposed',
                    endpoint=t, parameter='', payload='',
                    evidence=f'File accessible: {body[:60]}',
                    severity='Critical', cwe='CWE-538'
                ))

    def check_ssrf(self, base: str, param: str) -> None:
        payload = 'http://169.254.169.254/latest/meta-data'
        endpoint = f'{base}?{param}={urllib.parse.quote(payload)}'
        resp_endpoint = f'/api/ssrf?url=http://169.254.169.254/latest/meta-data'
        code_val, body = self._simulate_request(resp_endpoint)
        if code_val == 200 and 'ami' in body:
            self.findings.append(WebFinding(
                owasp_id='A10', name='SSRF — AWS Metadata',
                endpoint=base, parameter=param, payload=payload,
                evidence=f'Cloud metadata returned: {body[:60]}',
                severity='Critical', cwe='CWE-918'
            ))

    def run(self) -> List[WebFinding]:
        self.check_sqli('/login', 'user')
        self.check_xss('/search', 'q')
        self.check_idor('/api/user', 101)
        self.check_sensitive_files()
        self.check_ssrf('/api/ssrf', 'url')
        return self.findings

scanner = DASTScanner()
findings = scanner.run()
print(f'Found {len(findings)} vulnerabilities\n')
for f in findings:
    print(f'  [{f.severity}] {f.owasp_id} — {f.name}')
    print(f'    Endpoint: {f.endpoint}  Param: {f.parameter}')
    print(f'    Evidence: {f.evidence[:80]}')
    print(f'    CWE: {f.cwe}')

## Security Headers Analysis
HTTP security headers are a first line of defence: CSP prevents XSS injection, HSTS enforces HTTPS, X-Frame-Options prevents clickjacking. Missing headers are low-effort, high-impact fixes.

In [ ]:
SECURITY_HEADERS = {
    'Content-Security-Policy':      ('Critical', 'Prevents XSS and data injection'),
    'Strict-Transport-Security':    ('High',     'Enforces HTTPS, prevents protocol downgrade'),
    'X-Frame-Options':              ('Medium',   'Prevents clickjacking'),
    'X-Content-Type-Options':       ('Medium',   'Prevents MIME sniffing'),
    'Referrer-Policy':              ('Low',      'Controls referrer information leakage'),
    'Permissions-Policy':           ('Low',      'Restricts browser features'),
    'X-XSS-Protection':             ('Info',     'Legacy XSS filter (deprecated, use CSP)'),
}

def check_security_headers(response_headers: Dict[str, str]) -> List[Dict]:
    findings = []
    for header, (severity, desc) in SECURITY_HEADERS.items():
        if header not in response_headers:
            findings.append({'header': header, 'severity': severity,
                             'issue': 'Missing', 'description': desc})
        else:
            val = response_headers[header]
            # Check for weak CSP
            if header == 'Content-Security-Policy' and "'unsafe-inline'" in val:
                findings.append({'header': header, 'severity': 'High',
                                 'issue': "Contains 'unsafe-inline' — weakens XSS protection",
                                 'description': desc})
    return findings

# Simulate a typical misconfigured server
server_headers = {
    'Content-Type': 'text/html',
    'Server': 'Apache/2.4.41',
    'X-Frame-Options': 'SAMEORIGIN',
    'Content-Security-Policy': "default-src 'self' 'unsafe-inline'",
}

header_findings = check_security_headers(server_headers)
print(f'Security header analysis: {len(header_findings)} issues\n')
for f in header_findings:
    print(f'  [{f["severity"]}] {f["header"]}')
    print(f'    {f["issue"]} — {f["description"]}')

## Risk Scoring and Report Generation
Findings are deduplicated, CVSS-scored (approximated), and presented by priority. The report drives remediation order.

In [ ]:
SEVERITY_ORDER = {'Critical': 0, 'High': 1, 'Medium': 2, 'Low': 3, 'Info': 4}
CVSS_APPROX    = {'Critical': (9.0, 10.0), 'High': (7.0, 8.9), 'Medium': (4.0, 6.9), 'Low': (0.1, 3.9), 'Info': (0.0, 0.0)}

def generate_report(dast_findings: List[WebFinding], header_findings: List[Dict]) -> None:
    print('=' * 60)
    print('  WEB APPLICATION SECURITY ASSESSMENT REPORT')
    print('=' * 60)

    # Severity counts
    counts = {'Critical': 0, 'High': 0, 'Medium': 0, 'Low': 0, 'Info': 0}
    for f in dast_findings:
        counts[f.severity] += 1
    for f in header_findings:
        counts[f['severity']] += 1

    print('\nSummary:')
    for sev, cnt in sorted(counts.items(), key=lambda x: SEVERITY_ORDER[x[0]]):
        bar = '#' * cnt
        print(f'  {sev:<10} {cnt:>3}  {bar}')

    print('\nFindings by Priority:')
    all_f = [(f.severity, f.owasp_id, f.name, f.cwe) for f in dast_findings]
    all_f += [(f['severity'], 'A05', f['header'], 'CWE-16') for f in header_findings]
    all_f.sort(key=lambda x: SEVERITY_ORDER[x[0]])
    for sev, cat, name, cwe in all_f:
        lo, hi = CVSS_APPROX[sev]
        cvss_str = f'{lo:.1f}-{hi:.1f}' if hi > 0 else 'N/A'
        print(f'  [{sev:<8}] CVSS~{cvss_str:<8} {cat} {name} ({cwe})')

generate_report(findings, header_findings)